Step 1: Select segment to investigate. This section takes the target location like

TARGET_STATES = [
    ( 100, 0, 400),
    # (100, 400, 0),
    # (0, 0, 0),
]

and only uses selected segment for motion segmentation. The returned values need to be entered in the file below.


In [1]:
#!/usr/bin/env python3
import numpy as np
import pandas as pd

NORMAL_CSV = "/home/maurice/Documents/Master_Data/RRS/Co_motion/serial_normal.csv"
MOTOR_CSV  = "/home/maurice/Documents/Master_Data/RRS/Co_motion/serial_log_clean.csv"

# Enter motor states here
TARGET_STATES = [
    ( 100, 0, 400),
    # (100, 400, 0),
    # (0, 0, 0),
]

TOL_STEPS = 3

H, W = 480, 640
PAN_H, PAN_W = 1440, 1920

FX_PX = 1500.0
FY_PX = 1200.0

START_YAW_DEG = -45.0
START_PITCH_DEG = 0.0
START_ROLL_DEG = 0.0

USE_DESIRED_START = True

BASE_X = (PAN_W - W) * 0.5
BASE_Y = (PAN_H - H) * 0.5
CX = (W - 1) * 0.5
CY = (H - 1) * 0.5


def load_normals_csv(path):
    arr = np.loadtxt(path, delimiter=",", skiprows=1)
    t = arr[:, 0].astype(np.int64)
    n = arr[:, 1:4].astype(np.float64)

    idx = np.argsort(t)
    t = t[idx]
    n = n[idx]

    n /= np.linalg.norm(n, axis=1, keepdims=True) + 1e-12
    return t, n


def interp_normals(tN, nN, tq):
    tq = tq.astype(np.float64)
    tNf = tN.astype(np.float64)

    n = np.column_stack([
        np.interp(tq, tNf, nN[:, 0]),
        np.interp(tq, tNf, nN[:, 1]),
        np.interp(tq, tNf, nN[:, 2]),
    ])

    n /= np.linalg.norm(n, axis=1, keepdims=True) + 1e-12
    return n


def skew(v):
    x, y, z = v
    return np.array([
        [0.0, -z, y],
        [z, 0.0, -x],
        [-y, x, 0.0],
    ], dtype=np.float64)


def rot_axis_angle(axis, angle_deg):
    axis = np.asarray(axis, dtype=np.float64)
    axis /= np.linalg.norm(axis) + 1e-12

    a = np.deg2rad(angle_deg)
    K = skew(axis)

    return np.eye(3) + np.sin(a) * K + (1.0 - np.cos(a)) * (K @ K)


def rot_from_a_to_b(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)

    a /= np.linalg.norm(a) + 1e-12
    b /= np.linalg.norm(b) + 1e-12

    v = np.cross(a, b)
    c = float(np.dot(a, b))
    s = float(np.linalg.norm(v))

    if s < 1e-9:
        return np.eye(3)

    K = skew(v / s)
    return np.eye(3) + K * s + (K @ K) * (1.0 - c)


def desired_normal_from_yaw_pitch(yaw_deg, pitch_deg):
    yaw = np.deg2rad(yaw_deg)
    pitch = np.deg2rad(pitch_deg)

    n = np.array([np.sin(yaw), 0.0, np.cos(yaw)], dtype=np.float64)

    cp, sp = np.cos(pitch), np.sin(pitch)

    n = np.array([
        n[0],
        cp * n[1] - sp * n[2],
        sp * n[1] + cp * n[2],
    ])

    return n / (np.linalg.norm(n) + 1e-12)


def build_K():
    return np.array([
        [FX_PX, 0.0, CX],
        [0.0, FY_PX, CY],
        [0.0, 0.0, 1.0],
    ], dtype=np.float64)


def apply_H(points, Hmat):
    P = np.column_stack([points, np.ones(len(points), dtype=np.float64)])
    Q = (Hmat @ P.T).T

    Q[:, 0] /= Q[:, 2]
    Q[:, 1] /= Q[:, 2]

    return Q[:, :2]


def find_matching_motor_times(motor, target):
    A = motor["A"].to_numpy(np.int64)
    B = motor["B"].to_numpy(np.int64)
    C = motor["C"].to_numpy(np.int64)
    t = motor["t_us"].to_numpy(np.int64)

    ta, tb, tc = target

    match = (
        (np.abs(A - ta) <= TOL_STEPS) &
        (np.abs(B - tb) <= TOL_STEPS) &
        (np.abs(C - tc) <= TOL_STEPS)
    )

    return t[match]


def projected_roi_for_normals(normals, K, Kinv, n_ref, R_cal, R_roll):
    corners = np.array([
        [0, 0],
        [W - 1, 0],
        [W - 1, H - 1],
        [0, H - 1],
    ], dtype=np.float64)

    all_projected = []

    for n in normals:
        n_cur = R_cal @ n
        n_cur /= np.linalg.norm(n_cur) + 1e-12

        R_ref_to_cur = rot_from_a_to_b(n_ref, n_cur)
        R_cur_to_ref = R_ref_to_cur.T
        R_cur_to_ref = R_roll @ R_cur_to_ref

        H_cur_to_ref = K @ R_cur_to_ref @ Kinv

        warped = apply_H(corners, H_cur_to_ref)

        warped[:, 0] += BASE_X
        warped[:, 1] += BASE_Y

        all_projected.append(warped)

    all_projected = np.vstack(all_projected)

    x0 = int(np.floor(all_projected[:, 0].min()))
    x1 = int(np.ceil(all_projected[:, 0].max()))
    y0 = int(np.floor(all_projected[:, 1].min()))
    y1 = int(np.ceil(all_projected[:, 1].max()))

    x0 = max(0, x0)
    y0 = max(0, y0)
    x1 = min(PAN_W - 1, x1)
    y1 = min(PAN_H - 1, y1)

    return x0, y0, x1, y1


K = build_K()
Kinv = np.linalg.inv(K)

tN, nN = load_normals_csv(NORMAL_CSV)
motor = pd.read_csv(MOTOR_CSV).sort_values("t_us").reset_index(drop=True)

required = {"t_us", "A", "B", "C"}
if not required.issubset(motor.columns):
    raise RuntimeError(f"MOTOR_CSV must contain columns: {required}")

n_ref_meas = nN[0]

if USE_DESIRED_START:
    n_ref = desired_normal_from_yaw_pitch(START_YAW_DEG, START_PITCH_DEG)
    R_cal = rot_from_a_to_b(n_ref_meas, n_ref)
else:
    n_ref = n_ref_meas
    R_cal = np.eye(3)

R_roll = rot_axis_angle(n_ref, START_ROLL_DEG)

all_normals = []

for target in TARGET_STATES:
    times = find_matching_motor_times(motor, target)

    if len(times) == 0:
        print(f"Target {target}: not found")
        continue

    normals = interp_normals(tN, nN, times)
    all_normals.append(normals)

    x0, y0, x1, y1 = projected_roi_for_normals(
        normals, K, Kinv, n_ref, R_cal, R_roll
    )

    print()
    print("Target:", target)
    print("matching samples:", len(times))
    print(f"ROI_X0 = {x0}")
    print(f"ROI_Y0 = {y0}")
    print(f"ROI_W  = {x1 - x0 + 1}")
    print(f"ROI_H  = {y1 - y0 + 1}")

if len(all_normals) == 0:
    raise RuntimeError("No target states were found.")

all_normals = np.vstack(all_normals)

x0, y0, x1, y1 = projected_roi_for_normals(
    all_normals, K, Kinv, n_ref, R_cal, R_roll
)

print()
print("Combined ROI for all selected states:")
print(f"ROI_X0 = {x0}")
print(f"ROI_Y0 = {y0}")
print(f"ROI_W  = {x1 - x0 + 1}")
print(f"ROI_H  = {y1 - y0 + 1}")


Target: (100, 0, 400)
matching samples: 45
ROI_X0 = 998
ROI_Y0 = 408
ROI_W  = 730
ROI_H  = 525

Combined ROI for all selected states:
ROI_X0 = 998
ROI_Y0 = 408
ROI_W  = 730
ROI_H  = 525


Step 2: Take values for ROI from abobve to get "motion-filterd" output.

In [1]:
#!/usr/bin/env python3
import h5py
import numpy as np
import pandas as pd


# ============================================================
# Input / output
# ============================================================

EVENT_H5 = ( "/home/maurice/Documents/Master_Data/RRS/Co_motion/panorama_tilt_vectorized.h5"
            )

MOTOR_CSV = ("/home/maurice/Documents/Master_Data/RRS/Co_motion/serial_log_clean.csv"
)

OUT_H5 = (
    "/home/maurice/Documents/Master_Data/RRS/Co_motion/fan_100_400_0_segmented.h5"
)


# ============================================================
# Motor-position selection
# ============================================================

TARGET = (100, 400, 0)

# 1 = first visit, 2 = second visit, etc.
TARGET_VISIT = 2

# Motor coordinates may differ from TARGET by this many steps.
TOL_STEPS = 3

# Remove time from the beginning/end of the selected visit.
MARGIN_START_US = 0
MARGIN_END_US = 0


# ============================================================
# Region of interest
# ============================================================

PAN_H = 1440
PAN_W = 1920

ROI_X0 = 211
ROI_Y0 = 359
ROI_W  = 719
ROI_H  = 534

# ============================================================
# Spatiotemporal filtering parameters
# ============================================================

# Events are grouped into small space-time voxels.
#
# Smaller spatial cells:
#   - preserve more spatial detail;
#   - increase sensitivity to small motion;
#   - produce more occupied voxels.
#
# Larger spatial cells:
#   - tolerate spatial jitter;
#   - reduce noise;
#   - reduce spatial resolution.
SPATIAL_CELL_PX = 2

# Temporal voxel size.
#
# For a fast rotating fan, values around 25-200 us are reasonable
# starting points.
TEMPORAL_BIN_US = 50


# ------------------------------------------------------------
# Local density / isolated-noise rejection
# ------------------------------------------------------------

# Search radius for general local event support.
LOCAL_RADIUS_XY_CELLS = 1
LOCAL_RADIUS_T_BINS = 1

# Minimum number of events in the local space-time neighborhood.
#
# Increase this to remove isolated background activity.
# Decrease it if valid fast edges are being removed.
MIN_LOCAL_SUPPORT = 4


# ------------------------------------------------------------
# Motion-path support
# ------------------------------------------------------------

# A moving event should have support at a DIFFERENT spatial
# location in a nearby time bin.
#
# The radius should be large enough to cover the expected pixel
# displacement between temporal bins.
MOTION_RADIUS_XY_CELLS = 12
MOTION_RADIUS_T_BINS = 3

# Minimum number of events supporting motion into or out of the
# current voxel.
#
# This rejects events that repeatedly occur only at one location.
MIN_MOTION_SUPPORT = 2

# Require support in this many distinct nearby occupied voxels.
#
# This prevents one very large noisy voxel from satisfying the
# support threshold by itself.
MIN_MOTION_NEIGHBOR_VOXELS = 1


# ------------------------------------------------------------
# Persistent-location rejection
# ------------------------------------------------------------

# A spatial cell is considered persistent when it is active in a
# large fraction of all temporal bins.
#
# Example:
#   0.20 means a location is rejected if it is active in more
#   than 20% of the visit's temporal bins.
#
# Lower value:
#   more aggressive stationary/noise rejection.
#
# Higher value:
#   preserves objects that repeatedly revisit the same location,
#   such as rotating fan blades.
MAX_ACTIVE_BIN_FRACTION = 0.80

# Minimum number of active bins before persistence rejection is
# allowed. This avoids rejecting activity in very short visits.
MIN_ACTIVE_BINS_FOR_PERSISTENCE = 20

# Reject cells with an unusually large average number of events
# whenever they are active.
#
# Set to None to disable.
#
# This can help remove hot pixels or repeatedly firing sensor
# locations. A value around 5-20 is a possible starting range.
MAX_MEAN_EVENTS_PER_ACTIVE_BIN = None


# ------------------------------------------------------------
# Same-location dominance
# ------------------------------------------------------------

# A voxel can have local support but still be dominated by events
# occurring at exactly the same spatial cell.
#
# This threshold compares:
#
#     motion support / local support
#
# Larger values require a greater fraction of support to come
# from spatially displaced events.
MIN_MOTION_SUPPORT_RATIO = 0.10


# ------------------------------------------------------------
# Optional polarity requirements
# ------------------------------------------------------------

# Set True to require both positive and negative activity in the
# local trajectory neighborhood.
#
# This may remove noise, but it can also remove legitimate edges
# that primarily produce one polarity.
REQUIRE_BOTH_POLARITIES = False


# ============================================================
# Output and diagnostic options
# ============================================================

# Save rejected events as diagnostic datasets.
#
# For the smallest output file, leave this False.
SAVE_REJECTED_EVENTS = False

# Save sparse occupied-voxel diagnostics.
SAVE_VOXEL_DIAGNOSTICS = True

# Compression used for HDF5 datasets.
COMPRESSION = "gzip"


def find_target_interval(motor):
    """
    Find all consecutive motor-log intervals close to TARGET and
    return the requested visit.
    """
    t = motor["t_us"].to_numpy(np.int64)
    A = motor["A"].to_numpy(np.int64)
    B = motor["B"].to_numpy(np.int64)
    C = motor["C"].to_numpy(np.int64)

    target_a, target_b, target_c = TARGET

    match = (
        (np.abs(A - target_a) <= TOL_STEPS)
        & (np.abs(B - target_b) <= TOL_STEPS)
        & (np.abs(C - target_c) <= TOL_STEPS)
    )

    intervals = []
    i = 0

    while i < len(match):
        if not match[i]:
            i += 1
            continue

        start_index = i

        while i + 1 < len(match) and match[i + 1]:
            i += 1

        end_index = i
        intervals.append((start_index, end_index))
        i += 1

    if not intervals:
        raise RuntimeError(f"Target {TARGET} was never found.")

    print("\nDetected visits:")

    for visit_number, (start_index, end_index) in enumerate(
        intervals,
        start=1,
    ):
        raw_t0 = int(t[start_index])
        raw_t1 = int(t[end_index])

        cut_t0 = raw_t0 + MARGIN_START_US
        cut_t1 = raw_t1 - MARGIN_END_US

        print(
            f"Visit {visit_number}: "
            f"raw {raw_t0 / 1000:.3f} ms -> "
            f"{raw_t1 / 1000:.3f} ms, "
            f"cut {cut_t0 / 1000:.3f} ms -> "
            f"{cut_t1 / 1000:.3f} ms, "
            f"duration {(cut_t1 - cut_t0) / 1000:.3f} ms"
        )

    if TARGET_VISIT < 1 or TARGET_VISIT > len(intervals):
        raise RuntimeError(
            f"Requested TARGET_VISIT={TARGET_VISIT}, "
            f"but only {len(intervals)} visits exist."
        )

    start_index, end_index = intervals[TARGET_VISIT - 1]

    t0 = int(t[start_index]) + MARGIN_START_US
    t1 = int(t[end_index]) - MARGIN_END_US

    if t1 <= t0:
        raise RuntimeError(
            f"Selected visit {TARGET_VISIT} is too short after margins."
        )

    print("\nSelected interval:")
    print("Target:", TARGET)
    print("Visit:", TARGET_VISIT)
    print(f"Start:    {t0 / 1000:.3f} ms")
    print(f"End:      {t1 / 1000:.3f} ms")
    print(f"Duration: {(t1 - t0) / 1000:.3f} ms\n")

    return t0, t1, len(intervals)


def read_interval_events(h5, dataset_name, t0, t1):
    """
    Read all events in [t0, t1] from a timestamp-sorted HDF5
    dataset.

    Output timestamps are relative to t0.
    """
    dataset = h5[dataset_name]

    if dataset.shape[0] == 0:
        return np.zeros((0, 3), dtype=np.int64)

    timestamps = dataset[:, 2].astype(np.int64)

    start_index = np.searchsorted(
        timestamps,
        t0,
        side="left",
    )

    end_index = np.searchsorted(
        timestamps,
        t1,
        side="right",
    )

    events = np.asarray(
        dataset[start_index:end_index, :3],
        dtype=np.int64,
    )

    if events.shape[0] > 0:
        events[:, 2] -= t0

    return events


def crop_events(events):
    """
    Keep all events inside the selected ROI and convert their
    coordinates to ROI-local coordinates.
    """
    if events.shape[0] == 0:
        return events.copy()

    x = events[:, 0]
    y = events[:, 1]

    keep = (
        (x >= ROI_X0)
        & (x < ROI_X0 + ROI_W)
        & (y >= ROI_Y0)
        & (y < ROI_Y0 + ROI_H)
    )

    cropped = events[keep].copy()

    cropped[:, 0] -= ROI_X0
    cropped[:, 1] -= ROI_Y0

    return cropped


def combine_events(pos, neg):
    """
    Combine positive and negative events.

    Output columns:
        x, y, timestamp_us, polarity

    polarity:
        +1 = positive
        -1 = negative
    """
    arrays = []

    if pos.shape[0] > 0:
        pos_with_polarity = np.column_stack(
            [
                pos[:, :3],
                np.ones(pos.shape[0], dtype=np.int64),
            ]
        )
        arrays.append(pos_with_polarity)

    if neg.shape[0] > 0:
        neg_with_polarity = np.column_stack(
            [
                neg[:, :3],
                -np.ones(neg.shape[0], dtype=np.int64),
            ]
        )
        arrays.append(neg_with_polarity)

    if not arrays:
        return np.zeros((0, 4), dtype=np.int64)

    events = np.vstack(arrays)
    events = events[np.argsort(events[:, 2], kind="stable")]

    return events


def encode_voxel_key(tb, yc, xc, n_y_cells, n_x_cells):
    """
    Encode a 3D voxel coordinate into one int64 key.
    """
    return (
        (tb.astype(np.int64) * n_y_cells + yc.astype(np.int64))
        * n_x_cells
        + xc.astype(np.int64)
    )


def find_key_positions(sorted_keys, query_keys):
    """
    Find query keys in a sorted unique-key array.

    Returns:
        positions
        valid mask indicating exact matches
    """
    positions = np.searchsorted(sorted_keys, query_keys)

    valid = positions < len(sorted_keys)

    exact = np.zeros(len(query_keys), dtype=bool)

    if np.any(valid):
        exact[valid] = (
            sorted_keys[positions[valid]]
            == query_keys[valid]
        )

    return positions, exact


def calculate_neighbor_support(
    voxel_keys,
    voxel_counts,
    voxel_t,
    voxel_y,
    voxel_x,
    n_t_bins,
    n_y_cells,
    n_x_cells,
    radius_t,
    radius_xy,
    require_spatial_change,
    require_temporal_change,
):
    """
    Calculate event support and occupied-neighbor count around each
    occupied voxel.

    The calculation is sparse: it does not allocate a dense
    [time, y, x] array.
    """
    support = np.zeros(len(voxel_keys), dtype=np.int64)
    neighbor_voxel_count = np.zeros(
        len(voxel_keys),
        dtype=np.int32,
    )

    for dt in range(-radius_t, radius_t + 1):
        for dy in range(-radius_xy, radius_xy + 1):
            for dx in range(-radius_xy, radius_xy + 1):

                if require_spatial_change and dx == 0 and dy == 0:
                    continue

                if require_temporal_change and dt == 0:
                    continue

                neighbor_t = voxel_t + dt
                neighbor_y = voxel_y + dy
                neighbor_x = voxel_x + dx

                valid_coordinate = (
                    (neighbor_t >= 0)
                    & (neighbor_t < n_t_bins)
                    & (neighbor_y >= 0)
                    & (neighbor_y < n_y_cells)
                    & (neighbor_x >= 0)
                    & (neighbor_x < n_x_cells)
                )

                if not np.any(valid_coordinate):
                    continue

                source_indices = np.flatnonzero(valid_coordinate)

                query_keys = encode_voxel_key(
                    neighbor_t[valid_coordinate],
                    neighbor_y[valid_coordinate],
                    neighbor_x[valid_coordinate],
                    n_y_cells,
                    n_x_cells,
                )

                positions, exact = find_key_positions(
                    voxel_keys,
                    query_keys,
                )

                if not np.any(exact):
                    continue

                matched_source_indices = source_indices[exact]
                matched_positions = positions[exact]

                support[matched_source_indices] += (
                    voxel_counts[matched_positions]
                )

                neighbor_voxel_count[
                    matched_source_indices
                ] += 1

    return support, neighbor_voxel_count


def calculate_polarity_motion_support(
    voxel_keys_by_polarity,
    voxel_counts_by_polarity,
    query_t,
    query_y,
    query_x,
    n_t_bins,
    n_y_cells,
    n_x_cells,
):
    """
    Calculate moving-neighborhood support for one polarity.
    """
    support = np.zeros(len(query_t), dtype=np.int64)

    if len(voxel_keys_by_polarity) == 0:
        return support

    for dt in range(
        -MOTION_RADIUS_T_BINS,
        MOTION_RADIUS_T_BINS + 1,
    ):
        if dt == 0:
            continue

        for dy in range(
            -MOTION_RADIUS_XY_CELLS,
            MOTION_RADIUS_XY_CELLS + 1,
        ):
            for dx in range(
                -MOTION_RADIUS_XY_CELLS,
                MOTION_RADIUS_XY_CELLS + 1,
            ):
                if dx == 0 and dy == 0:
                    continue

                neighbor_t = query_t + dt
                neighbor_y = query_y + dy
                neighbor_x = query_x + dx

                valid_coordinate = (
                    (neighbor_t >= 0)
                    & (neighbor_t < n_t_bins)
                    & (neighbor_y >= 0)
                    & (neighbor_y < n_y_cells)
                    & (neighbor_x >= 0)
                    & (neighbor_x < n_x_cells)
                )

                if not np.any(valid_coordinate):
                    continue

                source_indices = np.flatnonzero(valid_coordinate)

                query_keys = encode_voxel_key(
                    neighbor_t[valid_coordinate],
                    neighbor_y[valid_coordinate],
                    neighbor_x[valid_coordinate],
                    n_y_cells,
                    n_x_cells,
                )

                positions, exact = find_key_positions(
                    voxel_keys_by_polarity,
                    query_keys,
                )

                if np.any(exact):
                    matched_sources = source_indices[exact]
                    matched_positions = positions[exact]

                    support[matched_sources] += (
                        voxel_counts_by_polarity[
                            matched_positions
                        ]
                    )

    return support


def calculate_spatial_persistence(
    voxel_t,
    voxel_y,
    voxel_x,
    voxel_counts,
    n_t_bins,
    n_y_cells,
    n_x_cells,
):
    """
    Calculate how persistently each spatial cell is active over the
    complete selected visit.

    Returns values for every occupied voxel.
    """
    spatial_keys = (
        voxel_y.astype(np.int64) * n_x_cells
        + voxel_x.astype(np.int64)
    )

    unique_spatial_keys, spatial_inverse = np.unique(
        spatial_keys,
        return_inverse=True,
    )

    active_bin_counts = np.bincount(
        spatial_inverse,
        minlength=len(unique_spatial_keys),
    ).astype(np.int64)

    spatial_event_counts = np.bincount(
        spatial_inverse,
        weights=voxel_counts,
        minlength=len(unique_spatial_keys),
    ).astype(np.float64)

    active_fraction_by_cell = (
        active_bin_counts.astype(np.float64)
        / max(n_t_bins, 1)
    )

    mean_events_per_active_bin_by_cell = (
        spatial_event_counts
        / np.maximum(active_bin_counts, 1)
    )

    active_bins_per_voxel = active_bin_counts[spatial_inverse]

    active_fraction_per_voxel = (
        active_fraction_by_cell[spatial_inverse]
    )

    mean_events_per_active_bin_per_voxel = (
        mean_events_per_active_bin_by_cell[spatial_inverse]
    )

    return (
        active_bins_per_voxel,
        active_fraction_per_voxel,
        mean_events_per_active_bin_per_voxel,
    )


def filter_moving_events(events, duration_us):
    """
    Classify each occupied spatiotemporal voxel and retain only
    events belonging to moving voxels.
    """
    if events.shape[0] == 0:
        raise RuntimeError("No events were found in the selected ROI.")

    n_x_cells = int(np.ceil(ROI_W / SPATIAL_CELL_PX))
    n_y_cells = int(np.ceil(ROI_H / SPATIAL_CELL_PX))
    n_t_bins = max(
        1,
        int(np.ceil(duration_us / TEMPORAL_BIN_US)),
    )

    event_x_cell = (
        events[:, 0].astype(np.int64)
        // SPATIAL_CELL_PX
    )

    event_y_cell = (
        events[:, 1].astype(np.int64)
        // SPATIAL_CELL_PX
    )

    event_t_bin = (
        events[:, 2].astype(np.int64)
        // TEMPORAL_BIN_US
    )

    event_t_bin = np.clip(
        event_t_bin,
        0,
        n_t_bins - 1,
    )

    event_keys = encode_voxel_key(
        event_t_bin,
        event_y_cell,
        event_x_cell,
        n_y_cells,
        n_x_cells,
    )

    voxel_keys, event_to_voxel, voxel_counts = np.unique(
        event_keys,
        return_inverse=True,
        return_counts=True,
    )

    # Decode the occupied voxel coordinates.
    voxel_x = voxel_keys % n_x_cells

    temp = voxel_keys // n_x_cells
    voxel_y = temp % n_y_cells
    voxel_t = temp // n_y_cells

    print("\nSparse voxel grid:")
    print("Temporal bins:", n_t_bins)
    print("Spatial cells x:", n_x_cells)
    print("Spatial cells y:", n_y_cells)
    print("Occupied voxels:", len(voxel_keys))

    # General local event density.
    local_support, local_neighbor_voxels = (
        calculate_neighbor_support(
            voxel_keys=voxel_keys,
            voxel_counts=voxel_counts,
            voxel_t=voxel_t,
            voxel_y=voxel_y,
            voxel_x=voxel_x,
            n_t_bins=n_t_bins,
            n_y_cells=n_y_cells,
            n_x_cells=n_x_cells,
            radius_t=LOCAL_RADIUS_T_BINS,
            radius_xy=LOCAL_RADIUS_XY_CELLS,
            require_spatial_change=False,
            require_temporal_change=False,
        )
    )

    # Support that explicitly changes both spatial location and time.
    motion_support, motion_neighbor_voxels = (
        calculate_neighbor_support(
            voxel_keys=voxel_keys,
            voxel_counts=voxel_counts,
            voxel_t=voxel_t,
            voxel_y=voxel_y,
            voxel_x=voxel_x,
            n_t_bins=n_t_bins,
            n_y_cells=n_y_cells,
            n_x_cells=n_x_cells,
            radius_t=MOTION_RADIUS_T_BINS,
            radius_xy=MOTION_RADIUS_XY_CELLS,
            require_spatial_change=True,
            require_temporal_change=True,
        )
    )

    (
        active_bins,
        active_fraction,
        mean_events_per_active_bin,
    ) = calculate_spatial_persistence(
        voxel_t=voxel_t,
        voxel_y=voxel_y,
        voxel_x=voxel_x,
        voxel_counts=voxel_counts,
        n_t_bins=n_t_bins,
        n_y_cells=n_y_cells,
        n_x_cells=n_x_cells,
    )

    motion_support_ratio = (
        motion_support.astype(np.float64)
        / np.maximum(local_support, 1)
    )

    persistent_location = (
        (active_bins >= MIN_ACTIVE_BINS_FOR_PERSISTENCE)
        & (active_fraction > MAX_ACTIVE_BIN_FRACTION)
    )

    if MAX_MEAN_EVENTS_PER_ACTIVE_BIN is not None:
        persistent_location |= (
            mean_events_per_active_bin
            > MAX_MEAN_EVENTS_PER_ACTIVE_BIN
        )

    moving_voxel = (
        (local_support >= MIN_LOCAL_SUPPORT)
        & (motion_support >= MIN_MOTION_SUPPORT)
        & (
            motion_neighbor_voxels
            >= MIN_MOTION_NEIGHBOR_VOXELS
        )
        & (
            motion_support_ratio
            >= MIN_MOTION_SUPPORT_RATIO
        )
        & (~persistent_location)
    )

    positive_motion_support = None
    negative_motion_support = None

    if REQUIRE_BOTH_POLARITIES:
        event_polarity = events[:, 3]

        positive_event_keys = event_keys[event_polarity > 0]
        negative_event_keys = event_keys[event_polarity < 0]

        positive_keys, positive_counts = np.unique(
            positive_event_keys,
            return_counts=True,
        )

        negative_keys, negative_counts = np.unique(
            negative_event_keys,
            return_counts=True,
        )

        positive_motion_support = (
            calculate_polarity_motion_support(
                voxel_keys_by_polarity=positive_keys,
                voxel_counts_by_polarity=positive_counts,
                query_t=voxel_t,
                query_y=voxel_y,
                query_x=voxel_x,
                n_t_bins=n_t_bins,
                n_y_cells=n_y_cells,
                n_x_cells=n_x_cells,
            )
        )

        negative_motion_support = (
            calculate_polarity_motion_support(
                voxel_keys_by_polarity=negative_keys,
                voxel_counts_by_polarity=negative_counts,
                query_t=voxel_t,
                query_y=voxel_y,
                query_x=voxel_x,
                n_t_bins=n_t_bins,
                n_y_cells=n_y_cells,
                n_x_cells=n_x_cells,
            )
        )

        moving_voxel &= (
            (positive_motion_support > 0)
            & (negative_motion_support > 0)
        )

    moving_event_mask = moving_voxel[event_to_voxel]

    moving_events = events[moving_event_mask].copy()
    static_events = events[~moving_event_mask].copy()

    diagnostics = {
        "voxel_key": voxel_keys,
        "voxel_t_bin": voxel_t,
        "voxel_y_cell": voxel_y,
        "voxel_x_cell": voxel_x,
        "voxel_event_count": voxel_counts,
        "local_support": local_support,
        "local_neighbor_voxels": local_neighbor_voxels,
        "motion_support": motion_support,
        "motion_neighbor_voxels": motion_neighbor_voxels,
        "motion_support_ratio": motion_support_ratio,
        "active_bins": active_bins,
        "active_fraction": active_fraction,
        "mean_events_per_active_bin": (
            mean_events_per_active_bin
        ),
        "persistent_location": persistent_location,
        "moving_voxel": moving_voxel,
    }

    if positive_motion_support is not None:
        diagnostics["positive_motion_support"] = (
            positive_motion_support
        )
        diagnostics["negative_motion_support"] = (
            negative_motion_support
        )

    print("\nFiltering results:")
    print("Input events:", len(events))
    print("Moving events:", len(moving_events))
    print("Static/rejected events:", len(static_events))

    retained_percent = (
        100.0 * len(moving_events) / max(len(events), 1)
    )

    print(f"Retained: {retained_percent:.2f}%")
    print(
        "Moving occupied voxels:",
        int(np.count_nonzero(moving_voxel)),
        "/",
        len(moving_voxel),
    )
    print(
        "Persistent occupied voxels:",
        int(np.count_nonzero(persistent_location)),
    )

    return moving_events, static_events, diagnostics


def split_by_polarity(events):
    """
    Convert combined [x, y, t, polarity] events back to separate
    positive and negative [x, y, t] arrays.
    """
    if events.shape[0] == 0:
        empty = np.zeros((0, 3), dtype=np.int64)
        return empty.copy(), empty.copy()

    pos = events[events[:, 3] > 0, :3].copy()
    neg = events[events[:, 3] < 0, :3].copy()

    return pos, neg


def save_output(
    path,
    pos_moving,
    neg_moving,
    pos_static,
    neg_static,#
    diagnostics,
    t0,
    t1,
    total_visits,
):
    """
    Save every selected ROI event as either static or moving.

    Visualizer-compatible datasets:
        pos_static
        neg_static
        pos_moving
        neg_moving

    Moving-only aliases for later speed analysis:
        pos
        neg
    """
    duration_us = t1 - t0

    total_static = len(pos_static) + len(neg_static)
    total_moving = len(pos_moving) + len(neg_moving)
    total_events = total_static + total_moving

    with h5py.File(path, "w") as out:
        out.create_dataset("pos_static", data=pos_static.astype(np.int64, copy=False), compression=COMPRESSION)
        out.create_dataset("neg_static", data=neg_static.astype(np.int64, copy=False), compression=COMPRESSION)
        out.create_dataset("pos_moving", data=pos_moving.astype(np.int64, copy=False), compression=COMPRESSION)
        out.create_dataset("neg_moving", data=neg_moving.astype(np.int64, copy=False), compression=COMPRESSION)

        # Moving-only aliases for downstream speed-analysis scripts.
        out.create_dataset("pos", data=pos_moving.astype(np.int64, copy=False), compression=COMPRESSION)
        out.create_dataset("neg", data=neg_moving.astype(np.int64, copy=False), compression=COMPRESSION)

        if SAVE_VOXEL_DIAGNOSTICS:
            group = out.create_group("voxel_diagnostics")
            for name, values in diagnostics.items():
                values = np.asarray(values)
                if values.dtype == bool:
                    values = values.astype(np.uint8)
                group.create_dataset(name, data=values, compression=COMPRESSION)

        out.attrs["num_pos_static"] = len(pos_static)
        out.attrs["num_neg_static"] = len(neg_static)
        out.attrs["num_pos_moving"] = len(pos_moving)
        out.attrs["num_neg_moving"] = len(neg_moving)
        out.attrs["num_static_events"] = total_static
        out.attrs["num_moving_events"] = total_moving
        out.attrs["num_total_roi_events"] = total_events
        out.attrs["moving_fraction"] = total_moving / max(total_events, 1)

        out.attrs["target_A"] = TARGET[0]
        out.attrs["target_B"] = TARGET[1]
        out.attrs["target_C"] = TARGET[2]
        out.attrs["target_visit"] = TARGET_VISIT
        out.attrs["total_target_visits"] = total_visits
        out.attrs["source_t0_us"] = t0
        out.attrs["source_t1_us"] = t1
        out.attrs["duration_us"] = duration_us

        out.attrs["PAN_W"] = ROI_W
        out.attrs["PAN_H"] = ROI_H
        out.attrs["roi_x0"] = ROI_X0
        out.attrs["roi_y0"] = ROI_Y0
        out.attrs["roi_w"] = ROI_W
        out.attrs["roi_h"] = ROI_H
        out.attrs["margin_start_us"] = MARGIN_START_US
        out.attrs["margin_end_us"] = MARGIN_END_US

        out.attrs["spatial_cell_px"] = SPATIAL_CELL_PX
        out.attrs["temporal_bin_us"] = TEMPORAL_BIN_US
        out.attrs["local_radius_xy_cells"] = LOCAL_RADIUS_XY_CELLS
        out.attrs["local_radius_t_bins"] = LOCAL_RADIUS_T_BINS
        out.attrs["min_local_support"] = MIN_LOCAL_SUPPORT
        out.attrs["motion_radius_xy_cells"] = MOTION_RADIUS_XY_CELLS
        out.attrs["motion_radius_t_bins"] = MOTION_RADIUS_T_BINS
        out.attrs["min_motion_support"] = MIN_MOTION_SUPPORT
        out.attrs["min_motion_neighbor_voxels"] = MIN_MOTION_NEIGHBOR_VOXELS
        out.attrs["min_motion_support_ratio"] = MIN_MOTION_SUPPORT_RATIO
        out.attrs["max_active_bin_fraction"] = MAX_ACTIVE_BIN_FRACTION
        out.attrs["min_active_bins_for_persistence"] = MIN_ACTIVE_BINS_FOR_PERSISTENCE
        if MAX_MEAN_EVENTS_PER_ACTIVE_BIN is not None:
            out.attrs["max_mean_events_per_active_bin"] = MAX_MEAN_EVENTS_PER_ACTIVE_BIN
        out.attrs["require_both_polarities"] = int(REQUIRE_BOTH_POLARITIES)
        out.attrs["timestamps_relative_to_visit_start"] = 1
        out.attrs["contains_static_and_moving"] = 1
        out.attrs["pos_neg_are_moving_only_aliases"] = 1


def main():
    motor = (
        pd.read_csv(MOTOR_CSV)
        .sort_values("t_us")
        .reset_index(drop=True)
    )

    required_columns = {"t_us", "A", "B", "C"}

    if not required_columns.issubset(motor.columns):
        raise RuntimeError(
            "Motor CSV must contain columns: "
            f"{required_columns}"
        )

    t0, t1, total_visits = find_target_interval(motor)
    duration_us = t1 - t0

    with h5py.File(EVENT_H5, "r") as h5:
        if "pos" not in h5 or "neg" not in h5:
            raise RuntimeError(
                "Input HDF5 file must contain 'pos' and 'neg'."
            )

        pos = read_interval_events(
            h5,
            "pos",
            t0,
            t1,
        )

        neg = read_interval_events(
            h5,
            "neg",
            t0,
            t1,
        )

    # These contain every event in the selected ROI and visit.
    pos = crop_events(pos)
    neg = crop_events(neg)

    print("All ROI positive events:", len(pos))
    print("All ROI negative events:", len(neg))
    print("All ROI events:", len(pos) + len(neg))

    events = combine_events(pos, neg)

    (
        moving_events,
        static_events,
        diagnostics,
    ) = filter_moving_events(
        events,
        duration_us,
    )

    pos_moving, neg_moving = split_by_polarity(moving_events)
    pos_static, neg_static = split_by_polarity(static_events)

    input_count = len(events)
    saved_count = (
        len(pos_static) + len(neg_static)
        + len(pos_moving) + len(neg_moving)
    )

    if saved_count != input_count:
        raise RuntimeError(
            "Static/moving classification lost events: "
            f"input={input_count}, saved={saved_count}"
        )

    save_output(
        path=OUT_H5,
        pos_moving=pos_moving,
        neg_moving=neg_moving,
        pos_static=pos_static,
        neg_static=neg_static,
        diagnostics=diagnostics,
        t0=t0,
        t1=t1,
        total_visits=total_visits,
    )

    print("\nSaved:", OUT_H5)
    print("Visit:", TARGET_VISIT, "/", total_visits)
    print("Duration:", duration_us / 1000, "ms")
    print("Static positive/negative:", len(pos_static), len(neg_static))
    print("Moving positive/negative:", len(pos_moving), len(neg_moving))
    print("Input/output event check:", input_count, saved_count)


if __name__ == "__main__":
    main()


Detected visits:
Visit 1: raw 2211.000 ms -> 2282.000 ms, cut 2211.000 ms -> 2282.000 ms, duration 71.000 ms
Visit 2: raw 6197.000 ms -> 6263.000 ms, cut 6197.000 ms -> 6263.000 ms, duration 66.000 ms

Selected interval:
Target: (100, 400, 0)
Visit: 2
Start:    6197.000 ms
End:      6263.000 ms
Duration: 66.000 ms

All ROI positive events: 68106
All ROI negative events: 76662
All ROI events: 144768

Sparse voxel grid:
Temporal bins: 1320
Spatial cells x: 360
Spatial cells y: 267
Occupied voxels: 139093

Filtering results:
Input events: 144768
Moving events: 2622
Static/rejected events: 142146
Retained: 1.81%
Moving occupied voxels: 2168 / 139093
Persistent occupied voxels: 0

Saved: /home/maurice/Documents/Master_Data/RRS/Co_motion/fan_100_400_0_segmented.h5
Visit: 2 / 2
Duration: 66.0 ms
Static positive/negative: 66751 75395
Moving positive/negative: 1355 1267
Input/output event check: 144768 144768


Step 3: Visualize the segmented output.

In [3]:
#!/usr/bin/env python3
import h5py
import numpy as np
import plotly.graph_objects as go


IN_H5 = (
   "/home/maurice/Documents/Master_Data/RRS/Co_motion/stitched_panorama_segmented.h5"
)

OUT_HTML = (
    "/home/maurice/Documents/Master_Data/RRS/Co_motion/segmented_full.html"
)


# Time is relative to the beginning of the selected motor interval.
TIME_START_US = 0
TIME_WINDOW_US = 50_000

# Maximum number of displayed occupied voxels.
MAX_POINTS = 120_000

# Spatial and temporal voxel dimensions.
VOXEL_SIZE_XY = 4
VOXEL_SIZE_T_US = 50

# Available modes:
#   "segmentation" -> static versus moving
#   "polarity"     -> positive versus negative
#   "time"         -> color by time
#   "count"        -> color by event count per voxel
COLOR_MODE = "segmentation"

# Which events to display:
#   "all"
#   "moving"
#   "static"
EVENT_SELECTION = "all"

# Marker sizes are scaled using the number of events in each voxel.
SCALE_MARKER_BY_COUNT = True
MARKER_SIZE_MIN = 2.0
MARKER_SIZE_MAX = 8.0

RANDOM_SEED = 123


def empty_events():
    """
    Return an empty event array with columns:

        x, y, timestamp, polarity, segmentation

    polarity:
        +1 = positive
        -1 = negative

    segmentation:
         1 = moving
         0 = static
    """
    return np.zeros((0, 5), dtype=np.int64)


def prepare_events(events, polarity, segmentation):
    """
    Convert an HDF5 event dataset into a common five-column format:

        x, y, timestamp, polarity, segmentation
    """
    events = np.asarray(events)

    if events.size == 0:
        return empty_events()

    if events.ndim != 2 or events.shape[1] < 3:
        raise RuntimeError(
            f"Expected event array with at least three columns, got {events.shape}"
        )

    n = events.shape[0]

    return np.column_stack(
        [
            events[:, 0].astype(np.int64),
            events[:, 1].astype(np.int64),
            events[:, 2].astype(np.int64),
            np.full(n, polarity, dtype=np.int64),
            np.full(n, segmentation, dtype=np.int64),
        ]
    )


def load_events(path):
    with h5py.File(path, "r") as h5:
        required = {
            "pos_static",
            "neg_static",
            "pos_moving",
            "neg_moving",
        }

        missing = required.difference(h5.keys())

        if missing:
            raise RuntimeError(
                "Missing required HDF5 datasets: "
                + ", ".join(sorted(missing))
            )

        pos_static = prepare_events(
            h5["pos_static"][:],
            polarity=1,
            segmentation=0,
        )

        neg_static = prepare_events(
            h5["neg_static"][:],
            polarity=-1,
            segmentation=0,
        )

        pos_moving = prepare_events(
            h5["pos_moving"][:],
            polarity=1,
            segmentation=1,
        )

        neg_moving = prepare_events(
            h5["neg_moving"][:],
            polarity=-1,
            segmentation=1,
        )

        roi_w = int(h5.attrs.get("roi_w", 0))
        roi_h = int(h5.attrs.get("roi_h", 0))
        source_t0_us = int(h5.attrs.get("source_t0_us", 0))
        duration_us = int(h5.attrs.get("duration_us", 0))

    arrays = [
        pos_static,
        neg_static,
        pos_moving,
        neg_moving,
    ]

    nonempty = [array for array in arrays if array.shape[0] > 0]

    if not nonempty:
        raise RuntimeError("All segmented event datasets are empty.")

    events = np.vstack(nonempty)
    events = events[np.argsort(events[:, 2])]

    if roi_w <= 0:
        roi_w = int(events[:, 0].max()) + 1

    if roi_h <= 0:
        roi_h = int(events[:, 1].max()) + 1

    if duration_us <= 0:
        duration_us = int(events[:, 2].max()) + 1

    return events, roi_w, roi_h, source_t0_us, duration_us


def select_events(events):
    """
    Select all events, only moving events, or only static events.
    """
    if EVENT_SELECTION == "all":
        return events

    if EVENT_SELECTION == "moving":
        return events[events[:, 4] == 1]

    if EVENT_SELECTION == "static":
        return events[events[:, 4] == 0]

    raise ValueError(
        "EVENT_SELECTION must be 'all', 'moving', or 'static'."
    )


def crop_time(events, duration_us):
    """
    Crop events using timestamps relative to the selected motor interval.
    """
    if events.shape[0] == 0:
        raise RuntimeError("No events available before time cropping.")

    timestamps = events[:, 2].astype(np.int64)

    if TIME_START_US is None:
        t0 = int(timestamps.min())
    else:
        t0 = int(TIME_START_US)

    if TIME_WINDOW_US is None:
        t1 = int(duration_us)
    else:
        t1 = t0 + int(TIME_WINDOW_US)

    mask = (timestamps >= t0) & (timestamps < t1)
    cropped = events[mask].copy()

    if cropped.shape[0] == 0:
        raise RuntimeError(
            f"No events found between {t0} us and {t1} us."
        )

    # Make displayed time relative to the beginning of the requested window.
    cropped[:, 2] -= t0

    return cropped, t0, t1


def voxelize(events):
    """
    Combine events into occupied voxels.

    Voxels are kept separate for:
        - polarity
        - segmentation class

    Therefore, positive/negative and static/moving events are not merged.
    """
    x = events[:, 0].astype(np.int64)
    y = events[:, 1].astype(np.int64)
    t = events[:, 2].astype(np.int64)
    polarity = events[:, 3].astype(np.int64)
    segmentation = events[:, 4].astype(np.int64)

    xv = x // VOXEL_SIZE_XY
    yv = y // VOXEL_SIZE_XY
    tv = t // VOXEL_SIZE_T_US

    keys = np.column_stack(
        [
            xv,
            yv,
            tv,
            polarity,
            segmentation,
        ]
    )

    unique_keys, counts = np.unique(
        keys,
        axis=0,
        return_counts=True,
    )

    x_plot = (
        unique_keys[:, 0] * VOXEL_SIZE_XY
        + VOXEL_SIZE_XY / 2.0
    )

    y_plot = (
        unique_keys[:, 1] * VOXEL_SIZE_XY
        + VOXEL_SIZE_XY / 2.0
    )

    t_plot = (
        unique_keys[:, 2] * VOXEL_SIZE_T_US
        + VOXEL_SIZE_T_US / 2.0
    ) / 1000.0

    polarity_plot = unique_keys[:, 3]
    segmentation_plot = unique_keys[:, 4]

    return (
        x_plot,
        y_plot,
        t_plot,
        polarity_plot,
        segmentation_plot,
        counts,
    )


def downsample(x, y, t, polarity, segmentation, counts):
    """
    Randomly downsample occupied voxels when the plot is too large.
    """
    n = len(x)

    if n <= MAX_POINTS:
        return x, y, t, polarity, segmentation, counts

    rng = np.random.default_rng(RANDOM_SEED)
    indices = rng.choice(
        n,
        size=MAX_POINTS,
        replace=False,
    )

    return (
        x[indices],
        y[indices],
        t[indices],
        polarity[indices],
        segmentation[indices],
        counts[indices],
    )


def marker_sizes(counts):
    """
    Scale marker size logarithmically using the event count in each voxel.
    """
    if not SCALE_MARKER_BY_COUNT:
        return np.full(len(counts), 3.0)

    counts = counts.astype(np.float64)
    transformed = np.log1p(counts)

    value_min = transformed.min()
    value_max = transformed.max()

    if value_max <= value_min:
        return np.full(len(counts), MARKER_SIZE_MIN)

    normalized = (
        (transformed - value_min)
        / (value_max - value_min)
    )

    return (
        MARKER_SIZE_MIN
        + normalized * (MARKER_SIZE_MAX - MARKER_SIZE_MIN)
    )


def add_segmentation_traces(
    fig,
    x,
    y,
    t,
    polarity,
    segmentation,
    counts,
    sizes,
):
    """
    Add separate static and moving traces.

    Separate traces give a clear legend and allow classes to be toggled.
    """
    classes = [
        (0, "Static", "royalblue"),
        (1, "Moving", "orangered"),
    ]

    for class_value, class_name, class_color in classes:
        mask = segmentation == class_value

        if not np.any(mask):
            continue

        hover_text = [
            (
                f"class={class_name}"
                f"<br>x={xi:.1f} px"
                f"<br>y={yi:.1f} px"
                f"<br>t={ti:.3f} ms"
                f"<br>polarity={int(pi):+d}"
                f"<br>count={int(ci)}"
            )
            for xi, yi, ti, pi, ci in zip(
                x[mask],
                y[mask],
                t[mask],
                polarity[mask],
                counts[mask],
            )
        ]

        fig.add_trace(
            go.Scatter3d(
                x=x[mask],
                y=t[mask],
                z=y[mask],
                mode="markers",
                name=class_name,
                marker=dict(
                    size=sizes[mask],
                    color=class_color,
                    opacity=0.72,
                ),
                text=hover_text,
                hoverinfo="text",
            )
        )


def add_polarity_traces(
    fig,
    x,
    y,
    t,
    polarity,
    segmentation,
    counts,
    sizes,
):
    """
    Add separate positive and negative traces.
    """
    classes = [
        (-1, "Negative", "blue"),
        (1, "Positive", "orange"),
    ]

    for polarity_value, polarity_name, polarity_color in classes:
        mask = polarity == polarity_value

        if not np.any(mask):
            continue

        class_names = np.where(
            segmentation[mask] == 1,
            "moving",
            "static",
        )

        hover_text = [
            (
                f"polarity={polarity_name}"
                f"<br>class={class_name}"
                f"<br>x={xi:.1f} px"
                f"<br>y={yi:.1f} px"
                f"<br>t={ti:.3f} ms"
                f"<br>count={int(ci)}"
            )
            for xi, yi, ti, class_name, ci in zip(
                x[mask],
                y[mask],
                t[mask],
                class_names,
                counts[mask],
            )
        ]

        fig.add_trace(
            go.Scatter3d(
                x=x[mask],
                y=t[mask],
                z=y[mask],
                mode="markers",
                name=polarity_name,
                marker=dict(
                    size=sizes[mask],
                    color=polarity_color,
                    opacity=0.72,
                ),
                text=hover_text,
                hoverinfo="text",
            )
        )


def add_continuous_trace(
    fig,
    x,
    y,
    t,
    polarity,
    segmentation,
    counts,
    sizes,
):
    """
    Add one continuous-color trace for time or event count.
    """
    if COLOR_MODE == "time":
        color = t
        colorscale = "Viridis"
        colorbar = dict(title="time [ms]")

    elif COLOR_MODE == "count":
        color = np.log1p(counts)
        colorscale = "Plasma"
        colorbar = dict(title="log(1 + count)")

    else:
        raise ValueError(
            "Continuous trace requires COLOR_MODE='time' or 'count'."
        )

    class_names = np.where(
        segmentation == 1,
        "moving",
        "static",
    )

    hover_text = [
        (
            f"class={class_name}"
            f"<br>x={xi:.1f} px"
            f"<br>y={yi:.1f} px"
            f"<br>t={ti:.3f} ms"
            f"<br>polarity={int(pi):+d}"
            f"<br>count={int(ci)}"
        )
        for xi, yi, ti, pi, class_name, ci in zip(
            x,
            y,
            t,
            polarity,
            class_names,
            counts,
        )
    ]

    fig.add_trace(
        go.Scatter3d(
            x=x,
            y=t,
            z=y,
            mode="markers",
            name=COLOR_MODE,
            marker=dict(
                size=sizes,
                color=color,
                colorscale=colorscale,
                opacity=0.72,
                colorbar=colorbar,
            ),
            text=hover_text,
            hoverinfo="text",
        )
    )


def main():
    (
        events,
        roi_w,
        roi_h,
        source_t0_us,
        duration_us,
    ) = load_events(IN_H5)

    print("total loaded events:", events.shape[0])
    print("static events:", np.count_nonzero(events[:, 4] == 0))
    print("moving events:", np.count_nonzero(events[:, 4] == 1))
    print("positive events:", np.count_nonzero(events[:, 3] == 1))
    print("negative events:", np.count_nonzero(events[:, 3] == -1))

    events = select_events(events)
    events, window_t0, window_t1 = crop_time(
        events,
        duration_us,
    )

    print()
    print("event selection:", EVENT_SELECTION)
    print("events in displayed window:", events.shape[0])
    print("relative time:", window_t0, "to", window_t1, "us")
    print(
        "absolute source time:",
        source_t0_us + window_t0,
        "to",
        source_t0_us + window_t1,
        "us",
    )
    print("ROI width:", roi_w)
    print("ROI height:", roi_h)

    (
        x,
        y,
        t,
        polarity,
        segmentation,
        counts,
    ) = voxelize(events)

    print("occupied voxels before downsampling:", len(x))

    (
        x,
        y,
        t,
        polarity,
        segmentation,
        counts,
    ) = downsample(
        x,
        y,
        t,
        polarity,
        segmentation,
        counts,
    )

    print("displayed voxels:", len(x))

    sizes = marker_sizes(counts)

    fig = go.Figure()

    if COLOR_MODE == "segmentation":
        add_segmentation_traces(
            fig,
            x,
            y,
            t,
            polarity,
            segmentation,
            counts,
            sizes,
        )

    elif COLOR_MODE == "polarity":
        add_polarity_traces(
            fig,
            x,
            y,
            t,
            polarity,
            segmentation,
            counts,
            sizes,
        )

    elif COLOR_MODE in {"time", "count"}:
        add_continuous_trace(
            fig,
            x,
            y,
            t,
            polarity,
            segmentation,
            counts,
            sizes,
        )

    else:
        raise ValueError(
            "COLOR_MODE must be 'segmentation', 'polarity', "
            "'time', or 'count'."
        )

    displayed_window_ms = (
        window_t1 - window_t0
    ) / 1000.0

    title = (
        "Segmented event voxel grid"
        f"<br><sup>"
        f"selection={EVENT_SELECTION}, "
        f"color={COLOR_MODE}, "
        f"voxel={VOXEL_SIZE_XY}×{VOXEL_SIZE_XY} px × "
        f"{VOXEL_SIZE_T_US} us"
        f"</sup>"
    )

    fig.update_layout(
        title=title,
        scene=dict(
            # Axis mapping:
            # x-axis = image x
            # y-axis = relative time
            # z-axis = image y
            xaxis_title="x [ROI px]",
            yaxis_title="time [ms]",
            zaxis_title="y [ROI px]",

            xaxis=dict(
                range=[0, roi_w],
                backgroundcolor="rgb(245,245,245)",
                gridcolor="white",
            ),

            yaxis=dict(
                range=[0, displayed_window_ms],
                backgroundcolor="rgb(245,245,245)",
                gridcolor="white",
            ),

            # Reverse image y so the view follows image coordinates.
            zaxis=dict(
                range=[roi_h, 0],
                backgroundcolor="rgb(245,245,245)",
                gridcolor="white",
            ),

            aspectmode="manual",
            aspectratio=dict(
                x=3.0,
                y=1.5,
                z=2.0,
            ),

            camera=dict(
                eye=dict(
                    x=1.6,
                    y=-1.7,
                    z=1.1,
                )
            ),
        ),

        legend=dict(
            title="Event class",
            x=0.01,
            y=0.99,
        ),

        width=1400,
        height=900,
        margin=dict(
            l=20,
            r=20,
            t=100,
            b=20,
        ),
    )

    fig.write_html(
        OUT_HTML,
        include_plotlyjs=True,
        full_html=True,
        auto_open=False,
    )

    print()
    print("Saved:", OUT_HTML)


if __name__ == "__main__":
    main()

total loaded events: 974543
static events: 934582
moving events: 39961
positive events: 475750
negative events: 498793

event selection: all
events in displayed window: 711183
relative time: 0 to 50000 us
absolute source time: 0 to 50000 us
ROI width: 1920
ROI height: 1440
occupied voxels before downsampling: 584041
displayed voxels: 120000

Saved: /home/maurice/Documents/Master_Data/RRS/Co_motion/segmented_full.html


Step 4: stitch together each segmented h5 back together to a panorama.

In [2]:
#!/usr/bin/env python3
import h5py
import numpy as np

INPUTS = [
    {
        "path": "/home/maurice/Documents/Master_Data/RRS/Co_motion/fan_0_0_0_segmented.h5",
        "motor_location": (0, 0, 0),
    },
    {
        "path": "/home/maurice/Documents/Master_Data/RRS/Co_motion/fan_100_0_400_segmented.h5",
        "motor_location": (100, 0, 400),
    },
    {
        "path": "/home/maurice/Documents/Master_Data/RRS/Co_motion/fan_100_400_0_segmented.h5",
        "motor_location": (100, 400, 0),
    },
]

OUT_H5 = "/home/maurice/Documents/Master_Data/RRS/Co_motion/stitched_panorama_segmented.h5"

PAN_W = 1920
PAN_H = 1440

# True if segmentation shifted ROI coordinates to start at (0,0).
# False if the segmented H5 already stores original panorama coordinates.
COORDINATES_ARE_ROI_LOCAL = True

# False overlays all visits on the same relative-time axis.
# True places them consecutively in time.
CONCATENATE_IN_TIME = False

REMOVE_EXACT_DUPLICATES = True
MOVING_HAS_PRIORITY = True
COMPRESSION = "gzip"

DATASET_NAMES = (
    "pos_static",
    "neg_static",
    "pos_moving",
    "neg_moving",
)


def empty_events():
    return np.zeros((0, 3), dtype=np.int64)


def read_events(h5, name, path):
    if name not in h5:
        raise RuntimeError(f"{path}: missing required dataset '{name}'")

    events = np.asarray(h5[name])

    if events.size == 0:
        return empty_events()

    if events.ndim != 2 or events.shape[1] < 3:
        raise RuntimeError(
            f"{path}: dataset '{name}' must have shape [N, >=3], got {events.shape}"
        )

    return np.asarray(events[:, :3], dtype=np.int64)


def rows_as_void(array):
    array = np.ascontiguousarray(array)
    return array.view(
        np.dtype((np.void, array.dtype.itemsize * array.shape[1]))
    ).ravel()


def sort_events(events):
    if events.shape[0] == 0:
        return empty_events()
    return events[np.argsort(events[:, 2], kind="stable")]


def unique_events(events):
    if events.shape[0] == 0:
        return empty_events()

    events = np.asarray(events, dtype=np.int64)
    _, idx = np.unique(rows_as_void(events), return_index=True)
    return sort_events(events[np.sort(idx)])


def merge_arrays(arrays):
    nonempty = [a for a in arrays if a.shape[0] > 0]

    if not nonempty:
        return empty_events()

    merged = np.vstack(nonempty)

    if REMOVE_EXACT_DUPLICATES:
        return unique_events(merged)

    return sort_events(merged)


def remove_events_present_in(reference, candidates):
    if reference.shape[0] == 0 or candidates.shape[0] == 0:
        return candidates

    keep = ~np.isin(
        rows_as_void(candidates),
        rows_as_void(reference),
        assume_unique=False,
    )

    return sort_events(candidates[keep])


def restore_panorama_coordinates(events, roi_x0, roi_y0):
    if events.shape[0] == 0:
        return empty_events()

    restored = events.copy()
    restored[:, 0] += int(roi_x0)
    restored[:, 1] += int(roi_y0)
    return restored


def validate_panorama_bounds(events, name, path):
    if events.shape[0] == 0:
        return

    valid = (
        (events[:, 0] >= 0)
        & (events[:, 0] < PAN_W)
        & (events[:, 1] >= 0)
        & (events[:, 1] < PAN_H)
    )

    bad = int(np.count_nonzero(~valid))

    if bad:
        raise RuntimeError(
            f"{path}: {bad} events in '{name}' fall outside "
            f"{PAN_W}x{PAN_H}. Check COORDINATES_ARE_ROI_LOCAL."
        )


def main():
    collected = {name: [] for name in DATASET_NAMES}
    source_info = []
    cumulative_time_offset_us = 0

    for index, config in enumerate(INPUTS):
        path = config["path"]

        with h5py.File(path, "r") as h5:
            roi_x0 = int(h5.attrs.get("roi_x0", 0))
            roi_y0 = int(h5.attrs.get("roi_y0", 0))
            roi_w = int(h5.attrs.get("roi_w", PAN_W))
            roi_h = int(h5.attrs.get("roi_h", PAN_H))
            duration_us = int(h5.attrs.get("duration_us", 0))
            source_t0_us = int(h5.attrs.get("source_t0_us", 0))

            time_offset_us = (
                cumulative_time_offset_us if CONCATENATE_IN_TIME else 0
            )

            print(f"Input {index + 1}: {path}")
            print("  motor location:", config["motor_location"])
            print("  ROI:", (roi_x0, roi_y0, roi_w, roi_h))
            print("  time offset:", time_offset_us)

            for name in DATASET_NAMES:
                events = read_events(h5, name, path)

                if COORDINATES_ARE_ROI_LOCAL:
                    events = restore_panorama_coordinates(
                        events, roi_x0, roi_y0
                    )

                if events.shape[0] > 0 and time_offset_us:
                    events[:, 2] += time_offset_us

                validate_panorama_bounds(events, name, path)
                collected[name].append(events)
                print(f"  {name}: {len(events)}")

            source_info.append(
                {
                    "path": path,
                    "motor_location": config["motor_location"],
                    "roi_x0": roi_x0,
                    "roi_y0": roi_y0,
                    "roi_w": roi_w,
                    "roi_h": roi_h,
                    "duration_us": duration_us,
                    "source_t0_us": source_t0_us,
                    "time_offset_us": time_offset_us,
                }
            )

            if CONCATENATE_IN_TIME:
                if duration_us <= 0:
                    local_times = []
                    for name in DATASET_NAMES:
                        events = collected[name][-1]
                        if events.shape[0] > 0:
                            local_times.append(events[:, 2] - time_offset_us)

                    if local_times:
                        duration_us = int(np.concatenate(local_times).max()) + 1
                    else:
                        duration_us = 0

                cumulative_time_offset_us += duration_us

    stitched = {
        name: merge_arrays(arrays)
        for name, arrays in collected.items()
    }

    if MOVING_HAS_PRIORITY:
        stitched["pos_static"] = remove_events_present_in(
            stitched["pos_moving"], stitched["pos_static"]
        )
        stitched["neg_static"] = remove_events_present_in(
            stitched["neg_moving"], stitched["neg_static"]
        )

    nonempty = [a for a in stitched.values() if a.shape[0] > 0]

    if not nonempty:
        raise RuntimeError("All stitched datasets are empty.")

    all_events = np.vstack(nonempty)
    min_t = int(all_events[:, 2].min())
    max_t = int(all_events[:, 2].max())
    duration_us = max_t - min_t + 1

    with h5py.File(OUT_H5, "w") as out:
        for name in DATASET_NAMES:
            out.create_dataset(
                name,
                data=stitched[name],
                compression=COMPRESSION,
            )

        out.create_dataset(
            "pos",
            data=stitched["pos_moving"],
            compression=COMPRESSION,
        )
        out.create_dataset(
            "neg",
            data=stitched["neg_moving"],
            compression=COMPRESSION,
        )

        out.attrs["PAN_W"] = PAN_W
        out.attrs["PAN_H"] = PAN_H
        out.attrs["roi_x0"] = 0
        out.attrs["roi_y0"] = 0
        out.attrs["roi_w"] = PAN_W
        out.attrs["roi_h"] = PAN_H

        out.attrs["source_t0_us"] = min_t
        out.attrs["source_t1_us"] = max_t
        out.attrs["duration_us"] = duration_us

        out.attrs["coordinate_system"] = "panorama"
        out.attrs["number_of_inputs"] = len(INPUTS)
        out.attrs["coordinates_were_roi_local"] = int(
            COORDINATES_ARE_ROI_LOCAL
        )
        out.attrs["concatenated_in_time"] = int(CONCATENATE_IN_TIME)
        out.attrs["moving_has_priority"] = int(MOVING_HAS_PRIORITY)

        group = out.create_group("inputs")

        for index, info in enumerate(source_info):
            source = group.create_group(f"{index:02d}")
            source.attrs["path"] = info["path"]

            a, b, c = info["motor_location"]
            source.attrs["motor_A"] = a
            source.attrs["motor_B"] = b
            source.attrs["motor_C"] = c

            source.attrs["roi_x0"] = info["roi_x0"]
            source.attrs["roi_y0"] = info["roi_y0"]
            source.attrs["roi_w"] = info["roi_w"]
            source.attrs["roi_h"] = info["roi_h"]
            source.attrs["duration_us"] = info["duration_us"]
            source.attrs["source_t0_us"] = info["source_t0_us"]
            source.attrs["time_offset_us"] = info["time_offset_us"]

    print()
    print("Stitched output:")
    for name in DATASET_NAMES:
        print(f"  {name}: {len(stitched[name])}")
    print("Saved:", OUT_H5)


if __name__ == "__main__":
    main()

Input 1: /home/maurice/Documents/Master_Data/RRS/Co_motion/fan_0_0_0_segmented.h5
  motor location: (0, 0, 0)
  ROI: (637, 479, 646, 482)
  time offset: 0
  pos_static: 120139
  neg_static: 116892
  pos_moving: 2096
  neg_moving: 1868
Input 2: /home/maurice/Documents/Master_Data/RRS/Co_motion/fan_100_0_400_segmented.h5
  motor location: (100, 0, 400)
  ROI: (998, 408, 730, 525)
  time offset: 0
  pos_static: 269072
  neg_static: 286452
  pos_moving: 16416
  neg_moving: 16982
Input 3: /home/maurice/Documents/Master_Data/RRS/Co_motion/fan_100_400_0_segmented.h5
  motor location: (100, 400, 0)
  ROI: (211, 359, 719, 534)
  time offset: 0
  pos_static: 66751
  neg_static: 75395
  pos_moving: 1355
  neg_moving: 1267

Stitched output:
  pos_static: 455894
  neg_static: 478688
  pos_moving: 19856
  neg_moving: 20105
Saved: /home/maurice/Documents/Master_Data/RRS/Co_motion/stitched_panorama_segmented.h5


In [ ]:
#!/usr/bin/env python3

import h5py
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree


# ============================================================
# Input and output
# ============================================================

EVENT_H5 = (
    "/home/maurice/Documents/Master_Data/RRS/"
    "Co_motion/panorama_tilt_vectorized.h5"
)

MOTOR_CSV = (
    "/home/maurice/Documents/Master_Data/RRS/"
    "Co_motion/serial_log_clean.csv"
)

OUT_H5 = (
    "/home/maurice/Documents/Master_Data/RRS/"
    "Co_motion/fan_optical_flow_segmented.h5"
)


# ============================================================
# Motor interval
# ============================================================

TARGET = (100, 400, 0)
TARGET_VISIT = 2
TOL_STEPS = 3

MARGIN_START_US = 0
MARGIN_END_US = 0


# ============================================================
# Region of interest
# ============================================================

ROI_X0 = 211
ROI_Y0 = 359
ROI_W = 719
ROI_H = 534


# ============================================================
# Voxelization
# ============================================================

# Spatial dimensions of one voxel.
SPATIAL_CELL_PX = 2

# Temporal dimension of one voxel.
TEMPORAL_BIN_US = 50


# ============================================================
# Local optical-flow estimation
# ============================================================

# Neighborhood used for fitting the local x-y-t plane.
FLOW_RADIUS_XY_PX = 16
FLOW_RADIUS_T_US = 500

# Minimum occupied voxels required for a plane fit.
MIN_FLOW_NEIGHBORS = 8

# Neighborhood must cover at least this temporal span.
MIN_TEMPORAL_SPAN_US = 150

# Reject poorly fitted planes.
MAX_PLANE_RMSE_US = 100.0

# Minimum coefficient magnitude. A nearly horizontal plane in
# x-y-t does not provide a stable velocity estimate.
MIN_PLANE_GRADIENT = 1e-8


# ============================================================
# Motion classification
# ============================================================

# Minimum estimated speed.
MIN_SPEED_PX_PER_MS = 1.0

# Optional upper bound to reject numerically unstable estimates.
MAX_SPEED_PX_PER_MS = 5000.0

# Require velocity consistency with nearby estimates.
REQUIRE_VELOCITY_CONSISTENCY = True

# Maximum angular difference between a voxel velocity and the
# weighted average velocity in its neighborhood.
MAX_DIRECTION_DIFFERENCE_DEG = 45.0

# Minimum number of neighboring valid flow estimates required
# for the consistency test.
MIN_CONSISTENT_FLOW_NEIGHBORS = 3


# ============================================================
# Output
# ============================================================

COMPRESSION = "gzip"
SAVE_DIAGNOSTICS = True


def find_target_interval(motor):
    """
    Find consecutive motor-log intervals close to TARGET and
    return the requested visit.
    """
    t = motor["t_us"].to_numpy(np.int64)
    a = motor["A"].to_numpy(np.int64)
    b = motor["B"].to_numpy(np.int64)
    c = motor["C"].to_numpy(np.int64)

    target_a, target_b, target_c = TARGET

    match = (
        (np.abs(a - target_a) <= TOL_STEPS)
        & (np.abs(b - target_b) <= TOL_STEPS)
        & (np.abs(c - target_c) <= TOL_STEPS)
    )

    intervals = []
    index = 0

    while index < len(match):
        if not match[index]:
            index += 1
            continue

        start_index = index

        while index + 1 < len(match) and match[index + 1]:
            index += 1

        intervals.append((start_index, index))
        index += 1

    if not intervals:
        raise RuntimeError(f"Target motor position {TARGET} was not found.")

    print("Detected target visits:")

    for visit_number, (start_index, end_index) in enumerate(
        intervals,
        start=1,
    ):
        raw_t0 = int(t[start_index])
        raw_t1 = int(t[end_index])

        print(
            f"  Visit {visit_number}: "
            f"{raw_t0 / 1000:.3f} ms to "
            f"{raw_t1 / 1000:.3f} ms"
        )

    if not 1 <= TARGET_VISIT <= len(intervals):
        raise RuntimeError(
            f"TARGET_VISIT={TARGET_VISIT}, but only "
            f"{len(intervals)} visits were found."
        )

    start_index, end_index = intervals[TARGET_VISIT - 1]

    t0 = int(t[start_index]) + MARGIN_START_US
    t1 = int(t[end_index]) - MARGIN_END_US

    if t1 <= t0:
        raise RuntimeError("Selected motor interval is empty.")

    print()
    print("Selected visit:", TARGET_VISIT)
    print("Start:", t0, "us")
    print("End:", t1, "us")
    print("Duration:", (t1 - t0) / 1000.0, "ms")

    return t0, t1, len(intervals)


def read_interval_events(h5, dataset_name, t0, t1):
    """
    Read timestamp-sorted events inside [t0, t1].

    Returned timestamps are relative to t0.
    """
    dataset = h5[dataset_name]

    if dataset.shape[0] == 0:
        return np.zeros((0, 3), dtype=np.int64)

    timestamps = np.asarray(dataset[:, 2], dtype=np.int64)

    start_index = np.searchsorted(
        timestamps,
        t0,
        side="left",
    )

    end_index = np.searchsorted(
        timestamps,
        t1,
        side="right",
    )

    events = np.asarray(
        dataset[start_index:end_index, :3],
        dtype=np.int64,
    )

    if len(events) > 0:
        events[:, 2] -= t0

    return events


def crop_to_roi(events):
    """
    Crop events to the selected ROI and convert them to local
    ROI coordinates.
    """
    if len(events) == 0:
        return events.copy()

    x = events[:, 0]
    y = events[:, 1]

    mask = (
        (x >= ROI_X0)
        & (x < ROI_X0 + ROI_W)
        & (y >= ROI_Y0)
        & (y < ROI_Y0 + ROI_H)
    )

    cropped = events[mask].copy()

    cropped[:, 0] -= ROI_X0
    cropped[:, 1] -= ROI_Y0

    return cropped


def combine_events(pos, neg):
    """
    Combine positive and negative events.

    Columns:
        x, y, timestamp_us, polarity
    """
    arrays = []

    if len(pos) > 0:
        arrays.append(
            np.column_stack(
                [
                    pos,
                    np.ones(len(pos), dtype=np.int64),
                ]
            )
        )

    if len(neg) > 0:
        arrays.append(
            np.column_stack(
                [
                    neg,
                    -np.ones(len(neg), dtype=np.int64),
                ]
            )
        )

    if not arrays:
        return np.zeros((0, 4), dtype=np.int64)

    events = np.vstack(arrays)

    return events[
        np.argsort(events[:, 2], kind="stable")
    ]


def encode_voxel_key(t_bin, y_cell, x_cell, n_y, n_x):
    """
    Encode a three-dimensional voxel coordinate as one integer.
    """
    return (
        (t_bin.astype(np.int64) * n_y + y_cell.astype(np.int64))
        * n_x
        + x_cell.astype(np.int64)
    )


def voxelize_events(events, duration_us):
    """
    Group events into occupied spatiotemporal voxels.
    """
    n_x = int(np.ceil(ROI_W / SPATIAL_CELL_PX))
    n_y = int(np.ceil(ROI_H / SPATIAL_CELL_PX))
    n_t = max(
        1,
        int(np.ceil(duration_us / TEMPORAL_BIN_US)),
    )

    x_cell = events[:, 0] // SPATIAL_CELL_PX
    y_cell = events[:, 1] // SPATIAL_CELL_PX
    t_bin = events[:, 2] // TEMPORAL_BIN_US

    t_bin = np.clip(t_bin, 0, n_t - 1)

    event_keys = encode_voxel_key(
        t_bin,
        y_cell,
        x_cell,
        n_y,
        n_x,
    )

    (
        voxel_keys,
        event_to_voxel,
        voxel_counts,
    ) = np.unique(
        event_keys,
        return_inverse=True,
        return_counts=True,
    )

    voxel_x_cell = voxel_keys % n_x

    temporary = voxel_keys // n_x
    voxel_y_cell = temporary % n_y
    voxel_t_bin = temporary // n_y

    # Use voxel-center coordinates.
    voxel_x = (
        voxel_x_cell.astype(np.float64)
        * SPATIAL_CELL_PX
        + SPATIAL_CELL_PX / 2.0
    )

    voxel_y = (
        voxel_y_cell.astype(np.float64)
        * SPATIAL_CELL_PX
        + SPATIAL_CELL_PX / 2.0
    )

    voxel_t = (
        voxel_t_bin.astype(np.float64)
        * TEMPORAL_BIN_US
        + TEMPORAL_BIN_US / 2.0
    )

    return {
        "keys": voxel_keys,
        "x": voxel_x,
        "y": voxel_y,
        "t": voxel_t,
        "counts": voxel_counts,
        "event_to_voxel": event_to_voxel,
    }


def fit_local_flow(
    voxel_x,
    voxel_y,
    voxel_t,
    voxel_counts,
):
    """
    Estimate local normal flow by fitting

        t = a*x + b*y + c

    around every occupied voxel.

    Velocity is derived from the minimum-norm solution of

        a*v_x + b*v_y = 1.
    """
    n_voxels = len(voxel_x)

    velocity_x = np.full(n_voxels, np.nan)
    velocity_y = np.full(n_voxels, np.nan)
    speed = np.full(n_voxels, np.nan)
    rmse = np.full(n_voxels, np.nan)
    neighbor_count = np.zeros(n_voxels, dtype=np.int32)
    temporal_span = np.zeros(n_voxels, dtype=np.float64)
    valid_flow = np.zeros(n_voxels, dtype=bool)

    # Scale time so that cKDTree distances are numerically
    # comparable to pixel distances.
    time_scale = FLOW_RADIUS_T_US / FLOW_RADIUS_XY_PX

    tree_points = np.column_stack(
        [
            voxel_x,
            voxel_y,
            voxel_t / time_scale,
        ]
    )

    tree = cKDTree(tree_points)

    search_radius = np.sqrt(
        2.0 * FLOW_RADIUS_XY_PX**2
        + (FLOW_RADIUS_T_US / time_scale) ** 2
    )

    for index in range(n_voxels):
        candidate_indices = tree.query_ball_point(
            tree_points[index],
            r=search_radius,
        )

        candidate_indices = np.asarray(
            candidate_indices,
            dtype=np.int64,
        )

        dx = np.abs(
            voxel_x[candidate_indices] - voxel_x[index]
        )

        dy = np.abs(
            voxel_y[candidate_indices] - voxel_y[index]
        )

        dt = np.abs(
            voxel_t[candidate_indices] - voxel_t[index]
        )

        inside = (
            (dx <= FLOW_RADIUS_XY_PX)
            & (dy <= FLOW_RADIUS_XY_PX)
            & (dt <= FLOW_RADIUS_T_US)
        )

        neighbors = candidate_indices[inside]
        neighbor_count[index] = len(neighbors)

        if len(neighbors) < MIN_FLOW_NEIGHBORS:
            continue

        neighborhood_t = voxel_t[neighbors]

        span = (
            neighborhood_t.max()
            - neighborhood_t.min()
        )

        temporal_span[index] = span

        if span < MIN_TEMPORAL_SPAN_US:
            continue

        # Center coordinates to improve numerical stability.
        x0 = voxel_x[index]
        y0 = voxel_y[index]
        t0 = voxel_t[index]

        local_x = voxel_x[neighbors] - x0
        local_y = voxel_y[neighbors] - y0
        local_t = voxel_t[neighbors] - t0

        design = np.column_stack(
            [
                local_x,
                local_y,
                np.ones(len(neighbors)),
            ]
        )

        weights = np.sqrt(
            voxel_counts[neighbors].astype(np.float64)
        )

        weighted_design = design * weights[:, None]
        weighted_time = local_t * weights

        try:
            coefficients, _, rank, _ = np.linalg.lstsq(
                weighted_design,
                weighted_time,
                rcond=None,
            )
        except np.linalg.LinAlgError:
            continue

        if rank < 3:
            continue

        a, b, c = coefficients

        predicted_t = design @ coefficients
        residual = local_t - predicted_t

        weighted_squared_error = np.sum(
            voxel_counts[neighbors] * residual**2
        )

        total_weight = np.sum(voxel_counts[neighbors])

        current_rmse = np.sqrt(
            weighted_squared_error / max(total_weight, 1)
        )

        rmse[index] = current_rmse

        gradient_squared = a * a + b * b

        if gradient_squared < MIN_PLANE_GRADIENT:
            continue

        # Minimum-norm normal-flow estimate.
        vx_px_per_us = a / gradient_squared
        vy_px_per_us = b / gradient_squared

        vx_px_per_ms = vx_px_per_us * 1000.0
        vy_px_per_ms = vy_px_per_us * 1000.0

        current_speed = np.hypot(
            vx_px_per_ms,
            vy_px_per_ms,
        )

        velocity_x[index] = vx_px_per_ms
        velocity_y[index] = vy_px_per_ms
        speed[index] = current_speed

        valid_flow[index] = (
            current_rmse <= MAX_PLANE_RMSE_US
            and MIN_SPEED_PX_PER_MS
            <= current_speed
            <= MAX_SPEED_PX_PER_MS
        )

    return {
        "velocity_x": velocity_x,
        "velocity_y": velocity_y,
        "speed": speed,
        "plane_rmse_us": rmse,
        "flow_neighbor_count": neighbor_count,
        "temporal_span_us": temporal_span,
        "valid_flow": valid_flow,
        "tree": tree,
        "tree_points": tree_points,
        "time_scale": time_scale,
        "search_radius": search_radius,
    }


def apply_velocity_consistency(
    voxel_x,
    voxel_y,
    voxel_t,
    flow,
):
    """
    Reject isolated velocity estimates whose direction is
    inconsistent with nearby valid estimates.
    """
    valid_flow = flow["valid_flow"].copy()

    if not REQUIRE_VELOCITY_CONSISTENCY:
        return valid_flow

    velocity_x = flow["velocity_x"]
    velocity_y = flow["velocity_y"]

    tree = flow["tree"]
    tree_points = flow["tree_points"]
    search_radius = flow["search_radius"]

    consistent = np.zeros(len(voxel_x), dtype=bool)

    maximum_angle = np.deg2rad(
        MAX_DIRECTION_DIFFERENCE_DEG
    )

    for index in np.flatnonzero(valid_flow):
        candidates = np.asarray(
            tree.query_ball_point(
                tree_points[index],
                r=search_radius,
            ),
            dtype=np.int64,
        )

        candidates = candidates[valid_flow[candidates]]

        if len(candidates) < MIN_CONSISTENT_FLOW_NEIGHBORS:
            continue

        dx = np.abs(voxel_x[candidates] - voxel_x[index])
        dy = np.abs(voxel_y[candidates] - voxel_y[index])
        dt = np.abs(voxel_t[candidates] - voxel_t[index])

        candidates = candidates[
            (dx <= FLOW_RADIUS_XY_PX)
            & (dy <= FLOW_RADIUS_XY_PX)
            & (dt <= FLOW_RADIUS_T_US)
        ]

        if len(candidates) < MIN_CONSISTENT_FLOW_NEIGHBORS:
            continue

        mean_vx = np.nanmedian(velocity_x[candidates])
        mean_vy = np.nanmedian(velocity_y[candidates])

        current_vector = np.array(
            [velocity_x[index], velocity_y[index]]
        )

        mean_vector = np.array([mean_vx, mean_vy])

        denominator = (
            np.linalg.norm(current_vector)
            * np.linalg.norm(mean_vector)
        )

        if denominator <= 0:
            continue

        cosine = np.clip(
            np.dot(current_vector, mean_vector)
            / denominator,
            -1.0,
            1.0,
        )

        angle = np.arccos(cosine)

        consistent[index] = angle <= maximum_angle

    return valid_flow & consistent


def segment_motion(events, duration_us):
    """
    Estimate local optical flow and classify events as moving or
    static/rejected.
    """
    if len(events) == 0:
        raise RuntimeError("No ROI events were found.")

    voxels = voxelize_events(events, duration_us)

    print()
    print("Occupied voxels:", len(voxels["keys"]))

    flow = fit_local_flow(
        voxel_x=voxels["x"],
        voxel_y=voxels["y"],
        voxel_t=voxels["t"],
        voxel_counts=voxels["counts"],
    )

    moving_voxel = apply_velocity_consistency(
        voxel_x=voxels["x"],
        voxel_y=voxels["y"],
        voxel_t=voxels["t"],
        flow=flow,
    )

    moving_event_mask = moving_voxel[
        voxels["event_to_voxel"]
    ]

    moving_events = events[moving_event_mask].copy()
    static_events = events[~moving_event_mask].copy()

    diagnostics = {
        "voxel_key": voxels["keys"],
        "voxel_x_px": voxels["x"],
        "voxel_y_px": voxels["y"],
        "voxel_t_us": voxels["t"],
        "voxel_event_count": voxels["counts"],
        "velocity_x_px_per_ms": flow["velocity_x"],
        "velocity_y_px_per_ms": flow["velocity_y"],
        "speed_px_per_ms": flow["speed"],
        "plane_rmse_us": flow["plane_rmse_us"],
        "flow_neighbor_count": flow["flow_neighbor_count"],
        "temporal_span_us": flow["temporal_span_us"],
        "valid_flow_before_consistency": flow["valid_flow"],
        "moving_voxel": moving_voxel,
    }

    print("Input events:", len(events))
    print("Moving events:", len(moving_events))
    print("Static/rejected events:", len(static_events))
    print(
        "Moving fraction:",
        len(moving_events) / max(len(events), 1),
    )
    print(
        "Moving voxels:",
        np.count_nonzero(moving_voxel),
        "/",
        len(moving_voxel),
    )

    return moving_events, static_events, diagnostics


def split_by_polarity(events):
    """
    Split [x, y, t, polarity] into separate positive and negative
    [x, y, t] arrays.
    """
    empty = np.zeros((0, 3), dtype=np.int64)

    if len(events) == 0:
        return empty.copy(), empty.copy()

    pos = events[events[:, 3] > 0, :3].copy()
    neg = events[events[:, 3] < 0, :3].copy()

    return pos, neg


def save_output(
    path,
    moving_events,
    static_events,
    diagnostics,
    t0,
    t1,
    total_visits,
):
    """
    Save the segmented event streams and flow diagnostics.
    """
    pos_moving, neg_moving = split_by_polarity(
        moving_events
    )

    pos_static, neg_static = split_by_polarity(
        static_events
    )

    with h5py.File(path, "w") as out:
        out.create_dataset(
            "pos_moving",
            data=pos_moving,
            compression=COMPRESSION,
        )

        out.create_dataset(
            "neg_moving",
            data=neg_moving,
            compression=COMPRESSION,
        )

        out.create_dataset(
            "pos_static",
            data=pos_static,
            compression=COMPRESSION,
        )

        out.create_dataset(
            "neg_static",
            data=neg_static,
            compression=COMPRESSION,
        )

        # Moving-only aliases.
        out.create_dataset(
            "pos",
            data=pos_moving,
            compression=COMPRESSION,
        )

        out.create_dataset(
            "neg",
            data=neg_moving,
            compression=COMPRESSION,
        )

        if SAVE_DIAGNOSTICS:
            group = out.create_group("flow_diagnostics")

            for name, values in diagnostics.items():
                values = np.asarray(values)

                if values.dtype == bool:
                    values = values.astype(np.uint8)

                group.create_dataset(
                    name,
                    data=values,
                    compression=COMPRESSION,
                )

        out.attrs["source_t0_us"] = t0
        out.attrs["source_t1_us"] = t1
        out.attrs["duration_us"] = t1 - t0

        out.attrs["target_A"] = TARGET[0]
        out.attrs["target_B"] = TARGET[1]
        out.attrs["target_C"] = TARGET[2]
        out.attrs["target_visit"] = TARGET_VISIT
        out.attrs["total_target_visits"] = total_visits

        out.attrs["roi_x0"] = ROI_X0
        out.attrs["roi_y0"] = ROI_Y0
        out.attrs["roi_w"] = ROI_W
        out.attrs["roi_h"] = ROI_H

        out.attrs["spatial_cell_px"] = SPATIAL_CELL_PX
        out.attrs["temporal_bin_us"] = TEMPORAL_BIN_US
        out.attrs["flow_radius_xy_px"] = FLOW_RADIUS_XY_PX
        out.attrs["flow_radius_t_us"] = FLOW_RADIUS_T_US
        out.attrs["min_flow_neighbors"] = MIN_FLOW_NEIGHBORS
        out.attrs["max_plane_rmse_us"] = MAX_PLANE_RMSE_US
        out.attrs["min_speed_px_per_ms"] = (
            MIN_SPEED_PX_PER_MS
        )

        total_events = len(moving_events) + len(static_events)

        out.attrs["num_moving_events"] = len(moving_events)
        out.attrs["num_static_events"] = len(static_events)
        out.attrs["num_total_events"] = total_events
        out.attrs["moving_fraction"] = (
            len(moving_events) / max(total_events, 1)
        )


def main():
    motor = (
        pd.read_csv(MOTOR_CSV)
        .sort_values("t_us")
        .reset_index(drop=True)
    )

    required_columns = {"t_us", "A", "B", "C"}

    if not required_columns.issubset(motor.columns):
        raise RuntimeError(
            "Motor CSV must contain t_us, A, B, and C."
        )

    t0, t1, total_visits = find_target_interval(motor)
    duration_us = t1 - t0

    with h5py.File(EVENT_H5, "r") as h5:
        if "pos" not in h5 or "neg" not in h5:
            raise RuntimeError(
                "Input HDF5 must contain 'pos' and 'neg'."
            )

        pos = read_interval_events(h5, "pos", t0, t1)
        neg = read_interval_events(h5, "neg", t0, t1)

    pos = crop_to_roi(pos)
    neg = crop_to_roi(neg)

    print()
    print("ROI positive events:", len(pos))
    print("ROI negative events:", len(neg))

    events = combine_events(pos, neg)

    moving_events, static_events, diagnostics = (
        segment_motion(
            events=events,
            duration_us=duration_us,
        )
    )

    save_output(
        path=OUT_H5,
        moving_events=moving_events,
        static_events=static_events,
        diagnostics=diagnostics,
        t0=t0,
        t1=t1,
        total_visits=total_visits,
    )

    print()
    print("Saved:", OUT_H5)


if __name__ == "__main__":
    main()